# Sequential multivariable AFT refitting for 41 headway distributions

This notebook independently reads:

1. the **41-model one-at-a-time covariate-screening workbook**, and
2. `data3.xlsx`.

For every pair–distribution model it:

- keeps screening candidates with one-at-a-time \(\Delta AIC \ge 2\) and
  covariate-effect **LR p < 0.05**;
- optionally requires the screening model to preserve the baseline
  goodness-of-fit acceptance decision;
- orders candidates from the largest to the smallest screening
  \(\Delta AIC\);
- adds them sequentially in a multivariable accelerated failure-time
  model,

  \[
  T_i=\exp(\mathbf z_i^\top\boldsymbol\beta)Y_i;
  \]

- retains an added covariate only when its **incremental**
  \(\Delta AIC \ge 2\) and its conditional LR p-value is below 0.05;
- performs final drop-one LR tests and removes terms that no longer
  remain significant after adjustment;
- calculates VIF only for the covariates retained in that particular
  pair–distribution model; and
- optionally performs refitted parametric-bootstrap KS and AD tests for
  every final model.

**Important interpretation:** β is a signed effect coefficient. The
threshold 0.05 belongs to the p-value testing β, not to the numerical
value of β itself.


In [1]:
# ============================================================
# 1. Imports, configuration, input discovery, and validation
# ============================================================

from pathlib import Path
import hashlib
import os
import time
import warnings

import numpy as np
import pandas as pd
from scipy import optimize, stats

from openpyxl import load_workbook
from openpyxl.formatting.rule import CellIsRule, FormulaRule
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

warnings.filterwarnings("ignore", category=RuntimeWarning)


# -------------------- user-adjustable settings --------------------

ALPHA_BETA = 0.05
MIN_SCREEN_DELTA_AIC = 2.0
MIN_STEP_DELTA_AIC = 2.0

# Earlier study rule: if the baseline model was GOF-accepted, a
# one-at-a-time covariate model must not turn it into a rejection.
REQUIRE_SCREEN_GOF_PRESERVED = True

# After forward entry, recheck every retained coefficient conditionally
# by drop-one LR tests and remove terms with p >= 0.05.
ENFORCE_FINAL_TERM_SIGNIFICANCE = True

VIF_WARNING_THRESHOLD = 5.0
VIF_SEVERE_THRESHOLD = 10.0

# This selection stage focuses on AIC, conditional p-values, and VIF,
# so final-model bootstrapping is off by default. Set the environment
# variable HEADWAY_N_BOOT_FINAL=499 when final KS/AD p-values are also
# required; every replicate refits the complete multivariable model.
N_BOOT_FINAL = int(os.environ.get("HEADWAY_N_BOOT_FINAL", "0"))
MIN_VALID_BOOT_FRACTION = 0.80
RANDOM_SEED = 20260730

SITE_ONE_LEVEL = "Tikatuli"
MIN_BINARY_GROUP_N = 5
BASELINE_AIC_TOLERANCE = 1e-3

# Optional testing hook. Leave at 0 to run all 41 models.
MODEL_LIMIT = int(os.environ.get("HEADWAY_MODEL_LIMIT", "0"))


HEADWAY = "Time_Headway"

COVARIATES = [
    "Target_Speed_km/hr",
    "Leading_Speed_km/hr",
    "Speed_Difference",
    "Occupancy",
    "Off_centeredness",
    "Site",
    "Flow_pcu/hr/m",
]

COVARIATE_LABELS = {
    "Target_Speed_km/hr": "Target Vehicle Speed",
    "Leading_Speed_km/hr": "Leading Vehicle Speed",
    "Speed_Difference": "Speed Difference",
    "Occupancy": "Occupancy",
    "Off_centeredness": "Off-centeredness",
    "Site": "Site",
    "Flow_pcu/hr/m": "Flow",
}

CONTINUOUS_COVARIATES = {
    "Target_Speed_km/hr",
    "Leading_Speed_km/hr",
    "Speed_Difference",
    "Flow_pcu/hr/m",
}

BINARY_COVARIATES = {
    "Occupancy",
    "Off_centeredness",
    "Site",
}


def to_boolean(series):
    """Convert Boolean, 0/1, and common text forms to bool."""
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y"])
    )


def stable_seed(*parts):
    """Stable seed independent of Python's randomized hash()."""
    token = "|".join(map(str, parts)).encode("utf-8")
    offset = int(hashlib.sha256(token).hexdigest()[:8], 16)
    return int((RANDOM_SEED + offset) % (2**32 - 1))


def is_41_model_screening_workbook(path):
    """Return True only for the required 41-model long-form workbook."""
    try:
        excel_file = pd.ExcelFile(path)
        if "S10_Full_screen" not in excel_file.sheet_names:
            return False
        inventory = pd.read_excel(
            path,
            sheet_name="S10_Full_screen",
            usecols=["Pair", "Distribution"],
        ).drop_duplicates()
        return len(inventory) == 41
    except Exception:
        return False


def discover_screening_workbook():
    explicit = os.environ.get("HEADWAY_SCREEN_PATH")
    candidates = []

    if explicit:
        candidates.append(Path(explicit))

    candidates.extend(
        [
            Path("D:/Headway/final run/tables/T4_41_Distribution_Covariate_Screening.xlsx"),
            Path("Tables/T4_41_Distribution_Covariate_Screening.xlsx"),
        ]
    )

    for folder in [Path("."), Path("Tables"), Path("upload"), Path("project_sources")]:
        if folder.exists():
            candidates.extend(sorted(folder.glob("*.xlsx")))

    seen = set()
    for candidate in candidates:
        resolved = str(candidate.resolve()) if candidate.exists() else str(candidate)
        if resolved in seen:
            continue
        seen.add(resolved)
        if candidate.exists() and is_41_model_screening_workbook(candidate):
            return candidate

    raise FileNotFoundError(
        "Could not find the 41-model screening workbook containing "
        "sheet 'S10_Full_screen'. Put it beside this notebook or set "
        "the HEADWAY_SCREEN_PATH environment variable."
    )


def discover_data_workbook():
    explicit = os.environ.get("HEADWAY_DATA_PATH")
    candidates = []
    if explicit:
        candidates.append(Path(explicit))
    candidates.extend(
        [
            Path("data3.xlsx"),
            Path("project_sources/12-data3.xlsx"),
            Path(r"D:\Headway\data3.xlsx"),
        ]
    )

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not find data3.xlsx. Put it beside this notebook or set "
        "the HEADWAY_DATA_PATH environment variable."
    )


SCREENING_PATH = discover_screening_workbook()
DATA_PATH = discover_data_workbook()

OUTPUT_DIR = Path(
    os.environ.get("HEADWAY_OUTPUT_DIR", "Tables")
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_XLSX = OUTPUT_DIR / "T5_41_Sequential_AFT_Selection_VIF.xlsx"


# -------------------- read screening workbook --------------------

screen = pd.read_excel(
    SCREENING_PATH,
    sheet_name="S10_Full_screen",
)
screen.columns = [str(column).strip() for column in screen.columns]

SCREEN_REQUIRED = [
    "Pair",
    "Distribution key",
    "Distribution",
    "Covariate",
    "Covariate label",
    "n",
    "Coding",
    "Estimable",
    "Baseline AIC",
    "ΔAIC",
    "β",
    "LR p",
    "GOF acceptance preserved",
    "Optimizer converged",
    "Baseline KS p",
    "Baseline AD p",
    "Baseline KS accepted",
    "Baseline AD accepted",
]

missing_screen_columns = [
    column for column in SCREEN_REQUIRED if column not in screen.columns
]
if missing_screen_columns:
    raise KeyError(
        "Missing screening columns: "
        + ", ".join(missing_screen_columns)
    )

for column in [
    "n",
    "Baseline AIC",
    "ΔAIC",
    "β",
    "LR p",
    "Baseline KS p",
    "Baseline AD p",
]:
    screen[column] = pd.to_numeric(screen[column], errors="coerce")

for column in [
    "Estimable",
    "GOF acceptance preserved",
    "Optimizer converged",
    "Baseline KS accepted",
    "Baseline AD accepted",
]:
    screen[column] = to_boolean(screen[column])

model_inventory = (
    screen[
        [
            "Pair",
            "Distribution key",
            "Distribution",
            "n",
            "Baseline AIC",
            "Baseline KS p",
            "Baseline AD p",
            "Baseline KS accepted",
            "Baseline AD accepted",
        ]
    ]
    .drop_duplicates(["Pair", "Distribution"], keep="first")
    .reset_index(drop=True)
)
model_inventory.insert(
    0,
    "Model order",
    np.arange(1, len(model_inventory) + 1),
)

if len(model_inventory) != 41:
    raise ValueError(
        f"Expected 41 pair–distribution models, found {len(model_inventory)}."
    )

model_order_lookup = model_inventory.set_index(
    ["Pair", "Distribution"]
)["Model order"]

screen["Model order"] = [
    model_order_lookup.loc[(pair, distribution)]
    for pair, distribution in zip(
        screen["Pair"],
        screen["Distribution"],
    )
]

screen["AIC condition"] = screen["ΔAIC"].ge(
    MIN_SCREEN_DELTA_AIC
)
screen["p-value condition"] = screen["LR p"].lt(ALPHA_BETA)
screen["Screening eligible"] = (
    screen["Estimable"]
    & screen["Optimizer converged"]
    & screen["AIC condition"]
    & screen["p-value condition"]
    & screen["β"].notna()
)

if REQUIRE_SCREEN_GOF_PRESERVED:
    screen["Screening eligible"] &= screen[
        "GOF acceptance preserved"
    ]


def screening_reason(row):
    reasons = []
    if not row["Estimable"]:
        reasons.append("not estimable")
    if not row["Optimizer converged"]:
        reasons.append("optimizer did not converge")
    if not row["AIC condition"]:
        reasons.append(f"screening ΔAIC < {MIN_SCREEN_DELTA_AIC:g}")
    if not row["p-value condition"]:
        reasons.append(f"screening LR p ≥ {ALPHA_BETA:g}")
    if not np.isfinite(row["β"]):
        reasons.append("β unavailable")
    if (
        REQUIRE_SCREEN_GOF_PRESERVED
        and not row["GOF acceptance preserved"]
    ):
        reasons.append("GOF acceptance not preserved")
    return "Eligible" if not reasons else "; ".join(reasons)


screen["Screening decision"] = screen.apply(
    screening_reason,
    axis=1,
)

screen["Screening rank"] = np.nan
eligible_mask = screen["Screening eligible"]
screen.loc[eligible_mask, "Screening rank"] = (
    screen.loc[eligible_mask]
    .groupby(["Pair", "Distribution"], sort=False)["ΔAIC"]
    .rank(method="first", ascending=False)
)


# -------------------- read raw data --------------------

raw_df = pd.read_excel(DATA_PATH, sheet_name=0)
raw_df.columns = [str(column).strip() for column in raw_df.columns]

DATA_REQUIRED = ["Pair", HEADWAY] + COVARIATES
missing_data_columns = [
    column for column in DATA_REQUIRED if column not in raw_df.columns
]
if missing_data_columns:
    raise KeyError(
        "Missing data columns: "
        + ", ".join(missing_data_columns)
    )

df = raw_df.copy()
for column in [HEADWAY] + sorted(CONTINUOUS_COVARIATES):
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Same complete-case rule as the screening notebook: comparisons within
# a pair use exactly the same observations.
valid = df[DATA_REQUIRED].notna().all(axis=1)
valid &= np.isfinite(df[HEADWAY]) & df[HEADWAY].gt(0)
for column in CONTINUOUS_COVARIATES:
    valid &= np.isfinite(df[column])

analysis_df = df.loc[valid].copy()

missing_pairs = sorted(
    set(model_inventory["Pair"])
    - set(analysis_df["Pair"].astype(str))
)
if missing_pairs:
    raise ValueError(
        "Pairs listed in the screening file are absent from data: "
        + ", ".join(missing_pairs)
    )

unexpected_sites = sorted(
    set(analysis_df["Site"].astype(str).unique())
    - {"Tikatuli", "Shahjahanpur"}
)
if unexpected_sites:
    raise ValueError(
        "Unexpected Site levels: "
        + ", ".join(unexpected_sites)
    )

actual_pair_n = analysis_df.groupby("Pair", observed=True).size()
model_inventory["Data n"] = model_inventory["Pair"].map(actual_pair_n)
model_inventory["n reconciled"] = (
    model_inventory["n"].astype(int)
    == model_inventory["Data n"].astype(int)
)
if not model_inventory["n reconciled"].all():
    bad = model_inventory.loc[
        ~model_inventory["n reconciled"],
        ["Pair", "n", "Data n"],
    ]
    raise ValueError(
        "Pair sample sizes do not match the screening workbook:\n"
        + bad.to_string(index=False)
    )

models_to_run = (
    model_inventory.head(MODEL_LIMIT).copy()
    if MODEL_LIMIT > 0
    else model_inventory.copy()
)

print("Screening workbook :", SCREENING_PATH)
print("Data workbook      :", DATA_PATH)
print("Output workbook    :", OUTPUT_XLSX)
print("Models to run      :", len(models_to_run))
print("Final bootstraps   :", N_BOOT_FINAL)
print("Complete data rows :", len(analysis_df))
print(
    "Eligible screening rows:",
    int(screen["Screening eligible"].sum()),
)


Screening workbook : D:\Headway\final run\tables\T4_41_Distribution_Covariate_Screening.xlsx
Data workbook      : D:\Headway\data3.xlsx
Output workbook    : Tables\T5_41_Sequential_AFT_Selection_VIF.xlsx
Models to run      : 41
Final bootstraps   : 0
Complete data rows : 898
Eligible screening rows: 61


In [2]:
# ============================================================
# 2. Distribution definitions and multivariable AFT likelihood
# ============================================================

DIST_SPECS = {
    "gengamma": {
        "label": "Generalized gamma",
        "distribution": stats.gengamma,
        "mode": "floc0",
    },
    "weibull_min": {
        "label": "Weibull",
        "distribution": stats.weibull_min,
        "mode": "floc0",
    },
    "pearson3": {
        "label": "Pearson type III",
        "distribution": stats.pearson3,
        "mode": "free",
    },
    "gamma": {
        "label": "Gamma",
        "distribution": stats.gamma,
        "mode": "floc0",
    },
    "lognorm": {
        "label": "Log-normal",
        "distribution": stats.lognorm,
        "mode": "floc0",
    },
    "invgauss": {
        "label": "Inverse Gaussian",
        "distribution": stats.invgauss,
        "mode": "floc0",
    },
}


def fit_baseline(key, x):
    """Baseline MLE using the original 41-model conventions."""
    x = np.asarray(x, dtype=float)
    specification = DIST_SPECS[key]
    distribution = specification["distribution"]

    if specification["mode"] == "free":
        parameters = tuple(
            np.asarray(distribution.fit(x), dtype=float)
        )
        k = len(parameters)
    else:
        parameters = tuple(
            np.asarray(
                distribution.fit(x, floc=0.0),
                dtype=float,
            )
        )
        k = len(parameters) - 1

    log_likelihood = float(
        np.sum(distribution.logpdf(x, *parameters))
    )
    if not np.isfinite(log_likelihood):
        raise FloatingPointError(
            f"Non-finite baseline likelihood for {key}"
        )

    theta = theta_from_model(
        key,
        parameters,
        betas=np.empty(0),
    )

    return {
        "parameters": parameters,
        "betas": np.empty(0),
        "covariates": [],
        "theta": theta,
        "logLik": log_likelihood,
        "k": int(k),
        "AIC": float(2 * k - 2 * log_likelihood),
        "converged": True,
        "optimizer": "scipy.fit",
        "optimizer_message": "Baseline MLE",
    }


def base_parameter_count(key):
    if key in {"gengamma", "pearson3"}:
        return 3
    if key in {
        "weibull_min",
        "gamma",
        "lognorm",
        "invgauss",
    }:
        return 2
    raise KeyError(key)


def theta_from_model(key, parameters, betas):
    """Transform parameters to a stable unconstrained vector."""
    p = tuple(float(value) for value in parameters)
    betas = np.asarray(betas, dtype=float)

    if key == "gengamma":
        a, c, _, scale = p
        base = [
            np.log(a),
            np.log(abs(c)),
            np.log(scale),
        ]
    elif key in {
        "weibull_min",
        "gamma",
        "lognorm",
        "invgauss",
    }:
        shape, _, scale = p
        base = [np.log(shape), np.log(scale)]
    elif key == "pearson3":
        skew, loc, scale = p
        base = [skew, loc, np.log(scale)]
    else:
        raise KeyError(key)

    return np.concatenate(
        [np.asarray(base, dtype=float), betas]
    )


def model_from_theta(key, theta, n_covariates):
    """Return SciPy parameters and β vector from optimizer θ."""
    theta = np.asarray(theta, dtype=float)
    base_k = base_parameter_count(key)

    if len(theta) != base_k + n_covariates:
        raise ValueError("Optimizer-vector length mismatch")

    if key == "gengamma":
        parameters = (
            float(np.exp(theta[0])),
            float(np.exp(theta[1])),
            0.0,
            float(np.exp(theta[2])),
        )
    elif key in {
        "weibull_min",
        "gamma",
        "lognorm",
        "invgauss",
    }:
        parameters = (
            float(np.exp(theta[0])),
            0.0,
            float(np.exp(theta[1])),
        )
    elif key == "pearson3":
        parameters = (
            float(theta[0]),
            float(theta[1]),
            float(np.exp(theta[2])),
        )
    else:
        raise KeyError(key)

    betas = np.asarray(theta[base_k:], dtype=float)
    return parameters, betas


def optimizer_bounds(key, n_covariates):
    log_positive = (-12.0, 12.0)

    if key == "gengamma":
        base = [
            log_positive,
            log_positive,
            (-20.0, 20.0),
        ]
    elif key in {
        "weibull_min",
        "gamma",
        "lognorm",
        "invgauss",
    }:
        base = [log_positive, (-20.0, 20.0)]
    elif key == "pearson3":
        base = [
            (-50.0, 50.0),
            (None, None),
            (-20.0, 20.0),
        ]
    else:
        raise KeyError(key)

    return base + [(-3.0, 3.0)] * n_covariates


def aft_negative_loglik(theta, key, x, Z):
    """
    Conditional negative log-likelihood under:
        T_i = exp(Z_i @ beta) * Y_i.
    """
    try:
        Z = np.asarray(Z, dtype=float)
        parameters, betas = model_from_theta(
            key,
            theta,
            Z.shape[1],
        )
        eta = np.clip(Z @ betas, -50.0, 50.0)
        adjusted = x * np.exp(-eta)
        distribution = DIST_SPECS[key]["distribution"]
        log_density = (
            distribution.logpdf(adjusted, *parameters)
            - eta
        )
    except Exception:
        return 1e100

    if not np.all(np.isfinite(log_density)):
        return 1e100
    return float(-np.sum(log_density))


def map_start_model(key, covariate_names, start_model):
    """Map a nested or reduced fitted model into a new design."""
    beta_map = dict(
        zip(
            start_model.get("covariates", []),
            np.asarray(
                start_model.get("betas", []),
                dtype=float,
            ),
        )
    )
    betas = np.array(
        [beta_map.get(name, 0.0) for name in covariate_names],
        dtype=float,
    )
    return theta_from_model(
        key,
        start_model["parameters"],
        betas,
    )


def fit_aft_multi(
    key,
    x,
    Z,
    covariate_names,
    start_model=None,
    robust=True,
    allow_nonpositive=False,
):
    """Fit a zero- or multi-covariate AFT model by maximum likelihood."""
    x = np.asarray(x, dtype=float)
    Z = np.asarray(Z, dtype=float)
    covariate_names = list(covariate_names)

    if Z.ndim != 2 or Z.shape[0] != len(x):
        raise ValueError("Z must have shape n × p")
    if Z.shape[1] != len(covariate_names):
        raise ValueError("Design columns and names do not match")
    if not np.all(np.isfinite(x)) or not np.all(np.isfinite(Z)):
        raise ValueError("Non-finite model input")
    if np.any(x <= 0) and not allow_nonpositive:
        raise ValueError("Headways must be strictly positive")

    n_covariates = Z.shape[1]
    baseline = fit_baseline(key, x)
    if n_covariates == 0:
        return baseline

    null_theta = theta_from_model(
        key,
        baseline["parameters"],
        np.zeros(n_covariates),
    )

    candidate_starts = [
        ("null start", null_theta, True),
    ]

    if start_model is not None:
        nested_theta = map_start_model(
            key,
            covariate_names,
            start_model,
        )
        candidate_starts.insert(
            0,
            ("nested start", nested_theta, True),
        )

    if robust:
        # A log-linear least-squares estimate is only a starting value;
        # the reported fit always comes from the stated AFT likelihood.
        try:
            slopes = np.linalg.lstsq(
                np.column_stack(
                    [np.ones(len(x)), Z]
                ),
                np.log(x),
                rcond=None,
            )[0][1:]
            slopes = np.clip(slopes, -1.5, 1.5)

            for multiplier in [1.0, -1.0, 0.5]:
                theta = null_theta.copy()
                theta[-n_covariates:] = (
                    multiplier * slopes
                )
                candidate_starts.append(
                    (
                        f"log-linear start × {multiplier:g}",
                        theta,
                        False,
                    )
                )
        except Exception:
            pass

    # Deduplicate numerically identical starts.
    starts = []
    for label, theta, is_nested in candidate_starts:
        if not any(
            np.allclose(theta, existing[1], rtol=0, atol=1e-10)
            for existing in starts
        ):
            starts.append((label, theta, is_nested))

    candidates = []
    for label, theta, is_nested in starts:
        # Heuristic starts are optimizer initials only. They must not
        # be treated as fitted solutions before optimization.
        if not is_nested:
            continue
        objective = aft_negative_loglik(
            theta,
            key,
            x,
            Z,
        )
        if np.isfinite(objective):
            candidates.append(
                {
                    "theta": np.asarray(theta, dtype=float),
                    "fun": float(objective),
                    "success": bool(is_nested),
                    "message": label,
                    "method": label,
                }
            )

    bounds = optimizer_bounds(key, n_covariates)

    for label, initial, _ in starts:
        try:
            result = optimize.minimize(
                aft_negative_loglik,
                initial,
                args=(key, x, Z),
                method="L-BFGS-B",
                bounds=bounds,
                options={
                    "maxiter": 5000 if robust else 1200,
                    "ftol": 1e-10 if robust else 1e-8,
                    "gtol": 1e-7 if robust else 1e-5,
                    "maxls": 50,
                },
            )
        except Exception:
            continue

        if np.isfinite(result.fun):
            candidates.append(
                {
                    "theta": np.asarray(
                        result.x,
                        dtype=float,
                    ),
                    "fun": float(result.fun),
                    "success": bool(result.success),
                    "message": (
                        f"{label}: {result.message}"
                    ),
                    "method": "L-BFGS-B",
                }
            )

    if not candidates:
        raise RuntimeError("No finite optimizer candidate")

    best = min(candidates, key=lambda item: item["fun"])

    # One fallback for a better-but-nonconverged L-BFGS-B solution.
    if robust and not best["success"]:
        try:
            fallback = optimize.minimize(
                aft_negative_loglik,
                best["theta"],
                args=(key, x, Z),
                method="Powell",
                bounds=bounds,
                options={
                    "maxiter": 5000,
                    "xtol": 1e-7,
                    "ftol": 1e-9,
                },
            )
            if np.isfinite(fallback.fun):
                candidates.append(
                    {
                        "theta": np.asarray(
                            fallback.x,
                            dtype=float,
                        ),
                        "fun": float(fallback.fun),
                        "success": bool(fallback.success),
                        "message": str(fallback.message),
                        "method": "Powell",
                    }
                )
                best = min(
                    candidates,
                    key=lambda item: item["fun"],
                )
        except Exception:
            pass

    parameters, betas = model_from_theta(
        key,
        best["theta"],
        n_covariates,
    )
    log_likelihood = float(-best["fun"])
    k = baseline["k"] + n_covariates

    return {
        "parameters": parameters,
        "betas": betas,
        "covariates": covariate_names,
        "theta": np.asarray(best["theta"], dtype=float),
        "logLik": log_likelihood,
        "k": int(k),
        "AIC": float(2 * k - 2 * log_likelihood),
        "converged": bool(best["success"]),
        "optimizer": best["method"],
        "optimizer_message": best["message"],
    }


def nested_lr_test(full_model, reduced_model, df_difference=1):
    """Likelihood-ratio comparison of two nested AFT models."""
    statistic = max(
        0.0,
        2.0
        * (
            float(full_model["logLik"])
            - float(reduced_model["logLik"])
        ),
    )
    p_value = float(
        stats.chi2.sf(statistic, df=df_difference)
    )
    return statistic, p_value


In [3]:
# ============================================================
# 3. Pair-specific design coding, forward selection, and VIF
# ============================================================

def encode_covariate(pair_data, covariate):
    """
    Encode one predictor exactly as in the screening notebook.

    Continuous predictors are standardized within the pair.
    """
    series = pair_data[covariate]

    if covariate in CONTINUOUS_COVARIATES:
        values = pd.to_numeric(
            series,
            errors="coerce",
        ).to_numpy(float)
        mean = float(np.mean(values))
        sd = float(np.std(values, ddof=1))
        if not np.isfinite(sd) or sd <= 1e-12:
            return (
                values * np.nan,
                "Constant within pair",
                False,
            )
        z = (values - mean) / sd
        note = (
            f"Within-pair z-score; "
            f"mean={mean:.6g}, SD={sd:.6g}"
        )
        return z, note, True

    if covariate in {"Occupancy", "Off_centeredness"}:
        if pd.api.types.is_bool_dtype(series):
            z = series.astype(float).to_numpy()
        else:
            normalized = (
                series.astype(str)
                .str.strip()
                .str.lower()
            )
            mapping = {
                "true": 1.0,
                "false": 0.0,
                "yes": 1.0,
                "no": 0.0,
                "1": 1.0,
                "0": 0.0,
            }
            z = normalized.map(mapping).to_numpy(float)

        if not np.all(np.isfinite(z)):
            return (
                z,
                "Unrecognized binary coding",
                False,
            )

        counts = pd.Series(z).value_counts()
        estimable = (
            set(counts.index) == {0.0, 1.0}
            and int(counts.min()) >= MIN_BINARY_GROUP_N
        )
        note = (
            f"False=0, True=1; "
            f"n0={(z == 0).sum()}, n1={(z == 1).sum()}"
        )
        return z, note, bool(estimable)

    if covariate == "Site":
        values = series.astype(str).to_numpy()
        z = (values == SITE_ONE_LEVEL).astype(float)
        counts = pd.Series(z).value_counts()
        estimable = (
            set(counts.index) == {0.0, 1.0}
            and int(counts.min()) >= MIN_BINARY_GROUP_N
        )
        note = (
            f"Shahjahanpur=0, {SITE_ONE_LEVEL}=1; "
            f"n0={(z == 0).sum()}, n1={(z == 1).sum()}"
        )
        return z, note, bool(estimable)

    raise KeyError(covariate)


def build_pair_design(pair_data):
    """Return the complete seven-covariate encoded design."""
    encoded = {}
    coding = {}
    estimability = {}

    for covariate in COVARIATES:
        z, note, estimable = encode_covariate(
            pair_data,
            covariate,
        )
        encoded[covariate] = np.asarray(z, dtype=float)
        coding[covariate] = note
        estimability[covariate] = bool(estimable)

    Z = np.column_stack(
        [encoded[covariate] for covariate in COVARIATES]
    )
    if not np.all(np.isfinite(Z)):
        raise ValueError(
            "The encoded design contains non-finite values."
        )

    design = pd.DataFrame(
        Z,
        columns=COVARIATES,
        index=pair_data.index,
    )
    return design, coding, estimability


def compute_vif(design, covariate_names):
    """
    Calculate VIF only on the supplied final covariate set.

    VIF_j = 1 / (1 - R_j²), where predictor j is regressed on
    the other retained predictors plus an intercept.
    """
    covariate_names = list(covariate_names)
    if not covariate_names:
        return {}
    if len(covariate_names) == 1:
        return {covariate_names[0]: 1.0}

    X = design[covariate_names].to_numpy(float)
    output = {}

    for column_index, covariate in enumerate(covariate_names):
        y = X[:, column_index]
        others = np.delete(X, column_index, axis=1)
        regressors = np.column_stack(
            [np.ones(len(y)), others]
        )
        coefficients = np.linalg.lstsq(
            regressors,
            y,
            rcond=None,
        )[0]
        residuals = y - regressors @ coefficients
        sse = float(np.sum(residuals**2))
        sst = float(np.sum((y - y.mean()) ** 2))

        if sst <= 1e-15:
            vif = np.inf
        else:
            r_squared = 1.0 - sse / sst
            r_squared = float(np.clip(r_squared, 0.0, 1.0))
            vif = (
                np.inf
                if 1.0 - r_squared <= 1e-12
                else 1.0 / (1.0 - r_squared)
            )

        output[covariate] = float(vif)

    return output


def vif_flag(vif):
    if not np.isfinite(vif) or vif >= VIF_SEVERE_THRESHOLD:
        return "Severe"
    if vif >= VIF_WARNING_THRESHOLD:
        return "Warning"
    return "Acceptable"


def design_matrix(design, covariate_names):
    covariate_names = list(covariate_names)
    if not covariate_names:
        return np.empty((len(design), 0), dtype=float)
    return design[covariate_names].to_numpy(float)


def drop_one_lr_tests(
    key,
    x,
    design,
    full_model,
    robust=True,
):
    """Conditional LR test for each term in a fitted final model."""
    tests = {}
    full_covariates = list(full_model["covariates"])

    for covariate in full_covariates:
        reduced_covariates = [
            name
            for name in full_covariates
            if name != covariate
        ]
        reduced_model = fit_aft_multi(
            key,
            x,
            design_matrix(design, reduced_covariates),
            reduced_covariates,
            start_model=full_model,
            robust=robust,
        )
        lr_statistic, p_value = nested_lr_test(
            full_model,
            reduced_model,
        )
        tests[covariate] = {
            "reduced_model": reduced_model,
            "LR χ²": lr_statistic,
            "LR p": p_value,
        }

    return tests


def fit_sequential_model(model_row):
    """Run screening-ordered forward AFT selection for one model."""
    pair = model_row["Pair"]
    distribution = model_row["Distribution"]
    key = model_row["Distribution key"]

    pair_data = analysis_df.loc[
        analysis_df["Pair"] == pair
    ].copy()
    x = pair_data[HEADWAY].to_numpy(float)
    design, coding, estimability = build_pair_design(
        pair_data
    )

    baseline = fit_baseline(key, x)
    screening_baseline_aic = float(
        model_row["Baseline AIC"]
    )
    baseline_difference = (
        baseline["AIC"] - screening_baseline_aic
    )

    if abs(baseline_difference) > BASELINE_AIC_TOLERANCE:
        raise ValueError(
            f"Baseline AIC failed to reconcile for "
            f"{pair} | {distribution}: "
            f"difference={baseline_difference:.6g}"
        )

    model_screen = screen.loc[
        (screen["Pair"] == pair)
        & (screen["Distribution"] == distribution)
    ].copy()

    candidates = (
        model_screen.loc[
            model_screen["Screening eligible"]
        ]
        .sort_values(
            ["ΔAIC", "Covariate"],
            ascending=[False, True],
            ignore_index=True,
        )
    )

    current = baseline
    selected = []
    step_rows = []

    # -------------------- ordered forward entry --------------------
    for forward_order, candidate_row in enumerate(
        candidates.itertuples(index=False),
        start=1,
    ):
        covariate = candidate_row.Covariate
        trial_covariates = selected + [covariate]
        trial = fit_aft_multi(
            key,
            x,
            design_matrix(design, trial_covariates),
            trial_covariates,
            start_model=current,
            robust=True,
        )

        incremental_delta_aic = (
            float(current["AIC"])
            - float(trial["AIC"])
        )
        lr_statistic, lr_p = nested_lr_test(
            trial,
            current,
        )
        candidate_beta = float(
            trial["betas"][
                trial_covariates.index(covariate)
            ]
        )
        boundary = abs(candidate_beta) >= 2.999

        passes = (
            trial["converged"]
            and np.isfinite(incremental_delta_aic)
            and incremental_delta_aic
            >= MIN_STEP_DELTA_AIC
            and np.isfinite(lr_p)
            and lr_p < ALPHA_BETA
            and not boundary
        )

        reasons = []
        if not trial["converged"]:
            reasons.append("optimizer did not converge")
        if (
            not np.isfinite(incremental_delta_aic)
            or incremental_delta_aic
            < MIN_STEP_DELTA_AIC
        ):
            reasons.append(
                f"incremental ΔAIC < "
                f"{MIN_STEP_DELTA_AIC:g}"
            )
        if not np.isfinite(lr_p) or lr_p >= ALPHA_BETA:
            reasons.append(
                f"conditional LR p ≥ {ALPHA_BETA:g}"
            )
        if boundary:
            reasons.append("β reached optimizer boundary")

        decision = (
            "Retained"
            if passes
            else "Rejected: " + "; ".join(reasons)
        )

        step_rows.append(
            {
                "Pair": pair,
                "Distribution": distribution,
                "Phase": "Forward entry",
                "Order": forward_order,
                "Candidate": covariate,
                "Candidate label": COVARIATE_LABELS[covariate],
                "Screening rank": int(
                    candidate_row._asdict()[
                        "Screening_rank"
                    ]
                )
                if "Screening_rank" in candidate_row._asdict()
                else forward_order,
                "Screening ΔAIC": float(
                    candidate_row._asdict()["ΔAIC"]
                ),
                "Screening LR p": float(
                    candidate_row._asdict()["LR_p"]
                )
                if "LR_p" in candidate_row._asdict()
                else float(
                    model_screen.loc[
                        model_screen["Covariate"] == covariate,
                        "LR p",
                    ].iloc[0]
                ),
                "Covariates before": (
                    ", ".join(selected)
                    if selected
                    else "(none)"
                ),
                "Trial covariates": ", ".join(
                    trial_covariates
                ),
                "AIC before": float(current["AIC"]),
                "Trial AIC": float(trial["AIC"]),
                "Incremental ΔAIC": incremental_delta_aic,
                "LR χ²": lr_statistic,
                "Conditional LR p": lr_p,
                "Candidate β": candidate_beta,
                "exp(β)": float(np.exp(candidate_beta)),
                "Optimizer converged": bool(
                    trial["converged"]
                ),
                "Decision": decision,
            }
        )

        if passes:
            selected = trial_covariates
            current = trial

    # -------------------- final conditional cleanup --------------------
    cleanup_order = 0
    if ENFORCE_FINAL_TERM_SIGNIFICANCE:
        while selected:
            tests = drop_one_lr_tests(
                key,
                x,
                design,
                current,
                robust=True,
            )
            non_significant = [
                (covariate, result)
                for covariate, result in tests.items()
                if (
                    not np.isfinite(result["LR p"])
                    or result["LR p"] >= ALPHA_BETA
                )
            ]
            if not non_significant:
                break

            covariate, result = max(
                non_significant,
                key=lambda item: (
                    np.inf
                    if not np.isfinite(item[1]["LR p"])
                    else item[1]["LR p"]
                ),
            )
            cleanup_order += 1
            before_covariates = list(selected)
            reduced = result["reduced_model"]
            selected = [
                name
                for name in selected
                if name != covariate
            ]

            step_rows.append(
                {
                    "Pair": pair,
                    "Distribution": distribution,
                    "Phase": "Backward cleanup",
                    "Order": cleanup_order,
                    "Candidate": covariate,
                    "Candidate label": COVARIATE_LABELS[covariate],
                    "Screening rank": np.nan,
                    "Screening ΔAIC": np.nan,
                    "Screening LR p": np.nan,
                    "Covariates before": ", ".join(
                        before_covariates
                    ),
                    "Trial covariates": (
                        ", ".join(selected)
                        if selected
                        else "(none)"
                    ),
                    "AIC before": float(current["AIC"]),
                    "Trial AIC": float(reduced["AIC"]),
                    "Incremental ΔAIC": (
                        float(current["AIC"])
                        - float(reduced["AIC"])
                    ),
                    "LR χ²": result["LR χ²"],
                    "Conditional LR p": result["LR p"],
                    "Candidate β": float(
                        current["betas"][
                            before_covariates.index(covariate)
                        ]
                    ),
                    "exp(β)": float(
                        np.exp(
                            current["betas"][
                                before_covariates.index(
                                    covariate
                                )
                            ]
                        )
                    ),
                    "Optimizer converged": bool(
                        reduced["converged"]
                    ),
                    "Decision": (
                        "Removed: final drop-one LR "
                        f"p ≥ {ALPHA_BETA:g}"
                    ),
                }
            )
            current = reduced

    # Final drop-one tests, coefficients, and pair-specific VIF.
    final_tests = (
        drop_one_lr_tests(
            key,
            x,
            design,
            current,
            robust=True,
        )
        if selected
        else {}
    )
    vif_values = compute_vif(design, selected)

    coefficient_rows = []
    for covariate, beta in zip(
        selected,
        current["betas"],
    ):
        beta = float(beta)
        effect_ratio = float(np.exp(beta))
        test = final_tests[covariate]
        coefficient_rows.append(
            {
                "Pair": pair,
                "Distribution": distribution,
                "Covariate": covariate,
                "Covariate label": COVARIATE_LABELS[covariate],
                "Coding": coding[covariate],
                "β": beta,
                "exp(β)": effect_ratio,
                "Percent headway change": (
                    100.0 * (effect_ratio - 1.0)
                ),
                "Effect direction": (
                    "Longer headway"
                    if beta > 0
                    else "Shorter headway"
                    if beta < 0
                    else "No change"
                ),
                "Drop-one LR χ²": test["LR χ²"],
                "Drop-one LR p": test["LR p"],
                "Final p < 0.05": (
                    test["LR p"] < ALPHA_BETA
                ),
                "VIF": vif_values[covariate],
                "VIF flag": vif_flag(
                    vif_values[covariate]
                ),
            }
        )

    vif_rows = [
        {
            "Pair": pair,
            "Distribution": distribution,
            "Final covariate set": (
                ", ".join(selected)
                if selected
                else "(none)"
            ),
            "Covariate": covariate,
            "Covariate label": COVARIATE_LABELS[covariate],
            "VIF": vif,
            "VIF flag": vif_flag(vif),
        }
        for covariate, vif in vif_values.items()
    ]

    baseline_audit = {
        "Pair": pair,
        "Distribution": distribution,
        "Distribution key": key,
        "n": len(x),
        "Screening baseline AIC": screening_baseline_aic,
        "Refitted baseline AIC": baseline["AIC"],
        "AIC difference": baseline_difference,
        "AIC reconciled": (
            abs(baseline_difference)
            <= BASELINE_AIC_TOLERANCE
        ),
    }

    return {
        "pair": pair,
        "distribution": distribution,
        "key": key,
        "x": x,
        "design": design,
        "coding": coding,
        "baseline": baseline,
        "final": current,
        "selected": selected,
        "screening_candidates": candidates,
        "step_rows": step_rows,
        "coefficient_rows": coefficient_rows,
        "vif_rows": vif_rows,
        "baseline_audit": baseline_audit,
    }


In [4]:
# ============================================================
# 4. Conditional PIT goodness of fit and final-model bootstrap
# ============================================================

def uniform_ks_ad(u):
    """KS D and AD A² for a sample tested against Uniform(0,1)."""
    u = np.sort(
        np.clip(
            np.asarray(u, dtype=float),
            1e-12,
            1 - 1e-12,
        )
    )
    n = len(u)
    i = np.arange(1, n + 1)

    ks_d = max(
        np.max(i / n - u),
        np.max(u - (i - 1) / n),
    )
    ad_a2 = (
        -n
        - np.sum(
            (2 * i - 1)
            * (
                np.log(u)
                + np.log(1 - u[::-1])
            )
        )
        / n
    )
    return float(ks_d), float(ad_a2)


def conditional_gof_statistics(key, x, Z, fitted_model):
    x = np.asarray(x, dtype=float)
    Z = np.asarray(Z, dtype=float)
    betas = np.asarray(fitted_model["betas"], dtype=float)
    eta = (
        np.clip(Z @ betas, -50.0, 50.0)
        if Z.shape[1]
        else np.zeros(len(x), dtype=float)
    )
    adjusted = x * np.exp(-eta)
    distribution = DIST_SPECS[key]["distribution"]
    u = distribution.cdf(
        adjusted,
        *fitted_model["parameters"],
    )
    if not np.all(np.isfinite(u)):
        raise FloatingPointError(
            "Non-finite conditional CDF values"
        )
    return uniform_ks_ad(u)


def bootstrap_final_gof(
    key,
    x,
    Z,
    covariate_names,
    fitted_model,
    n_boot,
    seed,
):
    """
    Parametric-bootstrap KS and AD tests.

    Covariates are held fixed and the complete final AFT model is
    refitted within every replicate.
    """
    x = np.asarray(x, dtype=float)
    Z = np.asarray(Z, dtype=float)
    n = len(x)
    distribution = DIST_SPECS[key]["distribution"]

    observed_ks, observed_ad = conditional_gof_statistics(
        key,
        x,
        Z,
        fitted_model,
    )

    if n_boot <= 0:
        return {
            "KS D": observed_ks,
            "KS p": np.nan,
            "AD A²": observed_ad,
            "AD p": np.nan,
            "Bootstrap valid": 0,
            "Bootstrap requested": 0,
            "Bootstrap status": "Not requested",
        }

    rng = np.random.default_rng(seed)
    betas = np.asarray(
        fitted_model["betas"],
        dtype=float,
    )
    eta = (
        np.clip(Z @ betas, -50.0, 50.0)
        if Z.shape[1]
        else np.zeros(n, dtype=float)
    )

    ks_exceedances = 0
    ad_exceedances = 0
    valid = 0

    for _ in range(n_boot):
        try:
            baseline_draw = np.asarray(
                distribution.rvs(
                    *fitted_model["parameters"],
                    size=n,
                    random_state=rng,
                ),
                dtype=float,
            )
            if (
                baseline_draw.shape != (n,)
                or not np.all(np.isfinite(baseline_draw))
            ):
                continue

            simulated_x = baseline_draw * np.exp(eta)
            if not np.all(np.isfinite(simulated_x)):
                continue
            if (
                key != "pearson3"
                and np.any(simulated_x <= 0)
            ):
                continue

            refit = fit_aft_multi(
                key,
                simulated_x,
                Z,
                covariate_names,
                start_model=fitted_model,
                robust=False,
                allow_nonpositive=(key == "pearson3"),
            )
            bootstrap_ks, bootstrap_ad = (
                conditional_gof_statistics(
                    key,
                    simulated_x,
                    Z,
                    refit,
                )
            )
            if not (
                np.isfinite(bootstrap_ks)
                and np.isfinite(bootstrap_ad)
            ):
                continue
        except Exception:
            continue

        valid += 1
        ks_exceedances += int(
            bootstrap_ks >= observed_ks
        )
        ad_exceedances += int(
            bootstrap_ad >= observed_ad
        )

    minimum_valid = int(
        np.ceil(MIN_VALID_BOOT_FRACTION * n_boot)
    )
    if valid < minimum_valid:
        return {
            "KS D": observed_ks,
            "KS p": np.nan,
            "AD A²": observed_ad,
            "AD p": np.nan,
            "Bootstrap valid": valid,
            "Bootstrap requested": n_boot,
            "Bootstrap status": (
                "INSUFFICIENT VALID REPLICATES: "
                f"{valid}/{n_boot}"
            ),
        }

    return {
        "KS D": observed_ks,
        "KS p": float(
            (1 + ks_exceedances) / (1 + valid)
        ),
        "AD A²": observed_ad,
        "AD p": float(
            (1 + ad_exceedances) / (1 + valid)
        ),
        "Bootstrap valid": valid,
        "Bootstrap requested": n_boot,
        "Bootstrap status": "OK",
    }


In [5]:
# ============================================================
# 5. Run all 41 sequential AFT analyses
# ============================================================

analysis_start = time.time()

fitted_results = []
all_steps = []
all_coefficients = []
all_vif = []
baseline_audits = []
model_summaries = []
gof_rows = []

for model_number, model_row in enumerate(
    models_to_run.to_dict("records"),
    start=1,
):
    pair = model_row["Pair"]
    distribution = model_row["Distribution"]
    print(
        f"[{model_number}/{len(models_to_run)}] "
        f"{pair} | {distribution}",
        end="",
    )

    model_start = time.time()
    result = fit_sequential_model(model_row)
    fitted_results.append(result)
    all_steps.extend(result["step_rows"])
    all_coefficients.extend(result["coefficient_rows"])
    all_vif.extend(result["vif_rows"])
    baseline_audits.append(result["baseline_audit"])

    final = result["final"]
    selected = result["selected"]
    Z_final = design_matrix(
        result["design"],
        selected,
    )

    gof = bootstrap_final_gof(
        result["key"],
        result["x"],
        Z_final,
        selected,
        final,
        n_boot=N_BOOT_FINAL,
        seed=stable_seed(
            pair,
            distribution,
            "final-gof",
        ),
    )

    baseline_ks_accepted = bool(
        model_row["Baseline KS accepted"]
    )
    baseline_ad_accepted = bool(
        model_row["Baseline AD accepted"]
    )

    final_ks_accepted = (
        bool(gof["KS p"] >= 0.05)
        if np.isfinite(gof["KS p"])
        else pd.NA
    )
    final_ad_accepted = (
        bool(gof["AD p"] >= 0.05)
        if np.isfinite(gof["AD p"])
        else pd.NA
    )

    ks_preserved = (
        (not baseline_ks_accepted)
        or (
            final_ks_accepted is not pd.NA
            and bool(final_ks_accepted)
        )
    )
    ad_preserved = (
        (not baseline_ad_accepted)
        or (
            final_ad_accepted is not pd.NA
            and bool(final_ad_accepted)
        )
    )
    gof_preserved = (
        bool(ks_preserved and ad_preserved)
        if N_BOOT_FINAL > 0
        and np.isfinite(gof["KS p"])
        and np.isfinite(gof["AD p"])
        else pd.NA
    )

    vif_values = [
        row["VIF"]
        for row in result["vif_rows"]
    ]
    max_vif = (
        max(vif_values)
        if vif_values
        else 1.0
    )

    eligible_order = (
        result["screening_candidates"]["Covariate"]
        .tolist()
    )

    model_summaries.append(
        {
            "Model order": model_row["Model order"],
            "Pair": pair,
            "Distribution": distribution,
            "Distribution key": result["key"],
            "n": len(result["x"]),
            "Eligible covariates in screening order": (
                ", ".join(eligible_order)
                if eligible_order
                else "(none)"
            ),
            "Number screening-eligible": len(eligible_order),
            "Final covariates": (
                ", ".join(selected)
                if selected
                else "(none)"
            ),
            "Number retained": len(selected),
            "Baseline k": result["baseline"]["k"],
            "Final k": final["k"],
            "Baseline logLik": result["baseline"]["logLik"],
            "Final logLik": final["logLik"],
            "Baseline AIC": result["baseline"]["AIC"],
            "Final AIC": final["AIC"],
            "Total ΔAIC": (
                result["baseline"]["AIC"]
                - final["AIC"]
            ),
            "Max VIF": max_vif,
            "VIF flag": vif_flag(max_vif),
            "Final KS D": gof["KS D"],
            "Final KS p": gof["KS p"],
            "Final AD A²": gof["AD A²"],
            "Final AD p": gof["AD p"],
            "Final GOF acceptance preserved": gof_preserved,
            "Optimizer converged": final["converged"],
            "Optimizer": final["optimizer"],
            "Model status": (
                "Final multivariable AFT"
                if selected
                else "Baseline retained: no covariate survived"
            ),
        }
    )

    gof_rows.append(
        {
            "Pair": pair,
            "Distribution": distribution,
            "Final covariates": (
                ", ".join(selected)
                if selected
                else "(none)"
            ),
            "Baseline KS p": model_row["Baseline KS p"],
            "Baseline KS accepted": baseline_ks_accepted,
            "Final KS D": gof["KS D"],
            "Final KS p": gof["KS p"],
            "Final KS accepted": final_ks_accepted,
            "KS acceptance preserved": (
                ks_preserved
                if N_BOOT_FINAL > 0
                else pd.NA
            ),
            "Baseline AD p": model_row["Baseline AD p"],
            "Baseline AD accepted": baseline_ad_accepted,
            "Final AD A²": gof["AD A²"],
            "Final AD p": gof["AD p"],
            "Final AD accepted": final_ad_accepted,
            "AD acceptance preserved": (
                ad_preserved
                if N_BOOT_FINAL > 0
                else pd.NA
            ),
            "Both acceptance decisions preserved": gof_preserved,
            "Bootstrap valid": gof["Bootstrap valid"],
            "Bootstrap requested": gof[
                "Bootstrap requested"
            ],
            "Bootstrap status": gof["Bootstrap status"],
        }
    )

    print(
        f" | retained={len(selected)}"
        f" | ΔAIC="
        f"{result['baseline']['AIC'] - final['AIC']:.3f}"
        f" | max VIF={max_vif:.3f}"
        f" | {time.time() - model_start:.1f}s"
    )


MODEL_SUMMARY = pd.DataFrame(model_summaries)
FORWARD_STEPS = pd.DataFrame(all_steps)
FINAL_COEFFICIENTS = pd.DataFrame(all_coefficients)
VIF_DETAILS = pd.DataFrame(all_vif)
BASELINE_AUDIT = pd.DataFrame(baseline_audits)
FINAL_GOF = pd.DataFrame(gof_rows)

if FORWARD_STEPS.empty:
    FORWARD_STEPS = pd.DataFrame(
        columns=[
            "Pair",
            "Distribution",
            "Phase",
            "Order",
            "Candidate",
            "Candidate label",
            "Screening rank",
            "Screening ΔAIC",
            "Screening LR p",
            "Covariates before",
            "Trial covariates",
            "AIC before",
            "Trial AIC",
            "Incremental ΔAIC",
            "LR χ²",
            "Conditional LR p",
            "Candidate β",
            "exp(β)",
            "Optimizer converged",
            "Decision",
        ]
    )

if FINAL_COEFFICIENTS.empty:
    FINAL_COEFFICIENTS = pd.DataFrame(
        columns=[
            "Pair",
            "Distribution",
            "Covariate",
            "Covariate label",
            "Coding",
            "β",
            "exp(β)",
            "Percent headway change",
            "Effect direction",
            "Drop-one LR χ²",
            "Drop-one LR p",
            "Final p < 0.05",
            "VIF",
            "VIF flag",
        ]
    )

if VIF_DETAILS.empty:
    VIF_DETAILS = pd.DataFrame(
        columns=[
            "Pair",
            "Distribution",
            "Final covariate set",
            "Covariate",
            "Covariate label",
            "VIF",
            "VIF flag",
        ]
    )

print(
    f"\nAnalysis completed in "
    f"{time.time() - analysis_start:.1f} seconds."
)


[1/41] PR_following_4W | Weibull | retained=0 | ΔAIC=0.000 | max VIF=1.000 | 0.0s
[2/41] PR_following_4W | Generalized gamma | retained=0 | ΔAIC=0.000 | max VIF=1.000 | 0.0s
[3/41] PR_following_4W | Pearson type III | retained=0 | ΔAIC=0.000 | max VIF=1.000 | 0.0s
[4/41] PR_following_4W | Gamma | retained=0 | ΔAIC=0.000 | max VIF=1.000 | 0.0s
[5/41] PR_following_4W | Log-normal | retained=1 | ΔAIC=2.917 | max VIF=1.000 | 0.0s
[6/41] PR_following_4W | Inverse Gaussian | retained=1 | ΔAIC=3.120 | max VIF=1.000 | 0.0s
[7/41] PR_following_MT_3W | Weibull | retained=1 | ΔAIC=16.078 | max VIF=1.000 | 0.0s
[8/41] PR_following_MT_3W | Generalized gamma | retained=1 | ΔAIC=16.037 | max VIF=1.000 | 0.1s
[9/41] PR_following_MT_3W | Pearson type III | retained=1 | ΔAIC=17.179 | max VIF=1.000 | 0.1s
[10/41] PR_following_MT_3W | Gamma | retained=2 | ΔAIC=17.711 | max VIF=1.044 | 0.2s
[11/41] PR_following_NMT_3W | Weibull | retained=0 | ΔAIC=0.000 | max VIF=1.000 | 0.0s
[12/41] PR_following_NMT_3W | 

In [6]:
# ============================================================
# 6. Build audit tables, matrices, and formatted Excel output
# ============================================================

METHOD_NOTES = pd.DataFrame(
    {
        "Item": [
            "Models",
            "Screening source",
            "Raw-data source",
            "AFT form",
            "Continuous-covariate coding",
            "Binary coding",
            "Site coding",
            "Screening eligibility",
            "Candidate order",
            "Forward retention",
            "Final coefficient check",
            "VIF scope",
            "VIF warning threshold",
            "VIF severe threshold",
            "Goodness-of-fit",
            "Bootstrap replicates",
            "Important β clarification",
        ],
        "Specification": [
            (
                f"{len(models_to_run)} of 41 "
                "pair–distribution models"
            ),
            str(SCREENING_PATH),
            str(DATA_PATH),
            "T_i = exp(z_i^T beta) Y_i",
            "Within-pair z-score",
            (
                "Occupancy and Off-centeredness: "
                "False=0, True=1"
            ),
            f"Shahjahanpur=0, {SITE_ONE_LEVEL}=1",
            (
                f"One-at-a-time ΔAIC ≥ "
                f"{MIN_SCREEN_DELTA_AIC:g}, "
                f"LR p < {ALPHA_BETA:g}, estimable, "
                "optimizer converged"
                + (
                    ", GOF acceptance preserved"
                    if REQUIRE_SCREEN_GOF_PRESERVED
                    else ""
                )
            ),
            (
                "Descending one-at-a-time ΔAIC within "
                "each pair–distribution"
            ),
            (
                f"Incremental ΔAIC ≥ "
                f"{MIN_STEP_DELTA_AIC:g} and "
                f"conditional LR p < {ALPHA_BETA:g}"
            ),
            (
                "Iterative drop-one LR cleanup at "
                f"p < {ALPHA_BETA:g}"
                if ENFORCE_FINAL_TERM_SIGNIFICANCE
                else "Not enforced"
            ),
            (
                "Only the covariates retained in the "
                "particular pair–distribution model"
            ),
            f"VIF ≥ {VIF_WARNING_THRESHOLD:g}",
            f"VIF ≥ {VIF_SEVERE_THRESHOLD:g} or infinite",
            (
                "Conditional PIT KS and AD with full "
                "model refitting in every bootstrap"
            ),
            str(N_BOOT_FINAL),
            (
                "The 0.05 threshold is applied to the "
                "covariate-effect p-value, not to β."
            ),
        ],
    }
)

SCREENING_AUDIT = screen.copy()
SCREENING_CANDIDATES = (
    screen.loc[
        screen["Screening eligible"],
        [
            "Model order",
            "Pair",
            "Distribution",
            "Covariate",
            "Covariate label",
            "Screening rank",
            "ΔAIC",
            "LR p",
            "β",
            "GOF acceptance preserved",
            "Coding",
        ],
    ]
    .sort_values(
        [
            "Model order",
            "Screening rank",
            "Covariate",
        ],
        ignore_index=True,
    )
)

selected_lookup = {
    (row["Pair"], row["Distribution"], row["Covariate"])
    for row in FINAL_COEFFICIENTS.to_dict("records")
}

selection_matrix_source = []
for model_row in models_to_run.to_dict("records"):
    row = {
        "Pair": model_row["Pair"],
        "Distribution": model_row["Distribution"],
    }
    for covariate in COVARIATES:
        row[COVARIATE_LABELS[covariate]] = (
            "Yes"
            if (
                model_row["Pair"],
                model_row["Distribution"],
                covariate,
            )
            in selected_lookup
            else "No"
        )
    selection_matrix_source.append(row)

SELECTION_MATRIX = pd.DataFrame(
    selection_matrix_source
)

beta_matrix_source = []
beta_lookup = {
    (
        row["Pair"],
        row["Distribution"],
        row["Covariate"],
    ): row["β"]
    for row in FINAL_COEFFICIENTS.to_dict("records")
}
for model_row in models_to_run.to_dict("records"):
    row = {
        "Pair": model_row["Pair"],
        "Distribution": model_row["Distribution"],
    }
    for covariate in COVARIATES:
        row[COVARIATE_LABELS[covariate]] = beta_lookup.get(
            (
                model_row["Pair"],
                model_row["Distribution"],
                covariate,
            ),
            np.nan,
        )
    beta_matrix_source.append(row)

BETA_MATRIX = pd.DataFrame(beta_matrix_source)


def style_excel_workbook(path):
    """Apply compact research-table formatting to every sheet."""
    workbook = load_workbook(path)

    navy = "17365D"
    blue = "D9EAF7"
    green = "E2F0D9"
    amber = "FFF2CC"
    red = "F4CCCC"
    white = "FFFFFF"
    grey = "E7E6E6"

    header_fill = PatternFill(
        "solid",
        fgColor=navy,
    )
    sub_fill = PatternFill(
        "solid",
        fgColor=blue,
    )
    green_fill = PatternFill(
        "solid",
        fgColor=green,
    )
    amber_fill = PatternFill(
        "solid",
        fgColor=amber,
    )
    red_fill = PatternFill(
        "solid",
        fgColor=red,
    )
    thin_grey = Side(
        style="thin",
        color="D9E1F2",
    )

    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        worksheet.sheet_view.showGridLines = False

        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = Font(
                color=white,
                bold=True,
            )
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )

        worksheet.row_dimensions[1].height = 34

        for row in worksheet.iter_rows(
            min_row=2,
            max_row=worksheet.max_row,
            min_col=1,
            max_col=worksheet.max_column,
        ):
            for cell in row:
                cell.alignment = Alignment(
                    vertical="top",
                    wrap_text=False,
                )
                cell.border = Border(
                    bottom=thin_grey,
                )

        for column_index in range(
            1,
            worksheet.max_column + 1,
        ):
            letter = get_column_letter(column_index)
            header = str(
                worksheet.cell(
                    row=1,
                    column=column_index,
                ).value
                or ""
            )
            values = [
                str(
                    worksheet.cell(
                        row=row_index,
                        column=column_index,
                    ).value
                    or ""
                )
                for row_index in range(
                    1,
                    min(worksheet.max_row, 250) + 1,
                )
            ]
            observed = max(
                [len(header)] + [len(value) for value in values]
            )
            if any(
                token in header
                for token in [
                    "Covariates",
                    "Decision",
                    "Coding",
                    "Specification",
                    "message",
                    "status",
                ]
            ):
                width = min(max(observed + 2, 18), 52)
            else:
                width = min(max(observed + 2, 11), 26)
            worksheet.column_dimensions[letter].width = width

        # Standard numeric formats based on header meaning.
        for cell in worksheet[1]:
            header = str(cell.value or "")
            if any(
                token in header
                for token in [
                    "AIC",
                    "logLik",
                    "β",
                    "exp(β)",
                    "VIF",
                    "KS D",
                    "AD A²",
                    "LR χ²",
                    "Percent",
                ]
            ):
                for data_cell in worksheet[
                    get_column_letter(cell.column)
                ][1:]:
                    data_cell.number_format = "0.0000"
            if header.endswith(" p") or "LR p" in header:
                for data_cell in worksheet[
                    get_column_letter(cell.column)
                ][1:]:
                    data_cell.number_format = "0.0000"

        headers = {
            str(cell.value): cell.column
            for cell in worksheet[1]
        }

        if "Decision" in headers and worksheet.max_row >= 2:
            column = get_column_letter(headers["Decision"])
            worksheet.conditional_formatting.add(
                f"{column}2:{column}{worksheet.max_row}",
                FormulaRule(
                    formula=[f'LEFT({column}2,8)="Retained"'],
                    fill=green_fill,
                ),
            )
            worksheet.conditional_formatting.add(
                f"{column}2:{column}{worksheet.max_row}",
                FormulaRule(
                    formula=[f'LEFT({column}2,8)="Rejected"'],
                    fill=red_fill,
                ),
            )

        if "VIF" in headers and worksheet.max_row >= 2:
            column = get_column_letter(headers["VIF"])
            worksheet.conditional_formatting.add(
                f"{column}2:{column}{worksheet.max_row}",
                CellIsRule(
                    operator="greaterThanOrEqual",
                    formula=[str(VIF_SEVERE_THRESHOLD)],
                    fill=red_fill,
                ),
            )
            worksheet.conditional_formatting.add(
                f"{column}2:{column}{worksheet.max_row}",
                CellIsRule(
                    operator="between",
                    formula=[
                        str(VIF_WARNING_THRESHOLD),
                        str(VIF_SEVERE_THRESHOLD),
                    ],
                    fill=amber_fill,
                ),
            )

        if "AIC reconciled" in headers:
            column = get_column_letter(
                headers["AIC reconciled"]
            )
            for row_index in range(
                2,
                worksheet.max_row + 1,
            ):
                cell = worksheet[
                    f"{column}{row_index}"
                ]
                cell.fill = (
                    green_fill
                    if cell.value is True
                    else red_fill
                )

    workbook.save(path)


export_sheets = {
    "S0_Method": METHOD_NOTES,
    "S1_Model_summary": MODEL_SUMMARY,
    "S2_Forward_steps": FORWARD_STEPS,
    "S3_Final_coefficients": FINAL_COEFFICIENTS,
    "S4_VIF_details": VIF_DETAILS,
    "S5_Final_GOF": FINAL_GOF,
    "S6_Screen_candidates": SCREENING_CANDIDATES,
    "S7_Screen_audit": SCREENING_AUDIT.drop(
        columns=[
            "Model order",
        ],
        errors="ignore",
    ),
    "S8_Baseline_audit": BASELINE_AUDIT,
    "S9_Selected_matrix": SELECTION_MATRIX,
    "S10_Beta_matrix": BETA_MATRIX,
}

with pd.ExcelWriter(
    OUTPUT_XLSX,
    engine="openpyxl",
) as writer:
    for sheet_name, table in export_sheets.items():
        table.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False,
        )

style_excel_workbook(OUTPUT_XLSX)

print("Wrote:", OUTPUT_XLSX)


Wrote: Tables\T5_41_Sequential_AFT_Selection_VIF.xlsx


In [7]:
# ============================================================
# 7. Compact results display and integrity checks
# ============================================================

if len(MODEL_SUMMARY) != len(models_to_run):
    raise AssertionError("Model-summary row count mismatch")

if not BASELINE_AUDIT["AIC reconciled"].all():
    raise AssertionError("At least one baseline AIC did not reconcile")

if not FINAL_COEFFICIENTS.empty:
    if not FINAL_COEFFICIENTS["Final p < 0.05"].all():
        raise AssertionError(
            "A retained final coefficient failed p < 0.05"
        )

    duplicated_coefficients = FINAL_COEFFICIENTS.duplicated(
        ["Pair", "Distribution", "Covariate"]
    )
    if duplicated_coefficients.any():
        raise AssertionError(
            "Duplicate final coefficient rows detected"
        )

accepted_forward = FORWARD_STEPS[
    FORWARD_STEPS["Decision"].eq("Retained")
]
if not accepted_forward.empty:
    if not (
        accepted_forward["Incremental ΔAIC"]
        .ge(MIN_STEP_DELTA_AIC)
        .all()
    ):
        raise AssertionError(
            "A retained forward step failed ΔAIC criterion"
        )
    if not (
        accepted_forward["Conditional LR p"]
        .lt(ALPHA_BETA)
        .all()
    ):
        raise AssertionError(
            "A retained forward step failed LR p criterion"
        )

print(
    "Models completed        :",
    len(MODEL_SUMMARY),
)
print(
    "Models with covariates  :",
    int(MODEL_SUMMARY["Number retained"].gt(0).sum()),
)
print(
    "Final retained terms    :",
    len(FINAL_COEFFICIENTS),
)
print(
    "Models with VIF warning :",
    int(
        MODEL_SUMMARY["Max VIF"]
        .ge(VIF_WARNING_THRESHOLD)
        .sum()
    ),
)
print(
    "Baseline AIC audit max  :",
    f"{BASELINE_AUDIT['AIC difference'].abs().max():.3g}",
)

display(
    MODEL_SUMMARY[
        [
            "Pair",
            "Distribution",
            "Eligible covariates in screening order",
            "Final covariates",
            "Baseline AIC",
            "Final AIC",
            "Total ΔAIC",
            "Max VIF",
            "VIF flag",
            "Final KS p",
            "Final AD p",
        ]
    ].round(4)
)

display(
    FINAL_COEFFICIENTS.round(
        {
            "β": 4,
            "exp(β)": 4,
            "Percent headway change": 2,
            "Drop-one LR χ²": 3,
            "Drop-one LR p": 4,
            "VIF": 3,
        }
    )
)


Models completed        : 41
Models with covariates  : 34
Final retained terms    : 50
Models with VIF warning : 0
Baseline AIC audit max  : 5.68e-14


,Pair,Distribution,Eligible covariates in screening order,Final covariates,Baseline AIC,Final AIC,Total ΔAIC,Max VIF,VIF flag,Final KS p,Final AD p
0,PR_following_4W,Weibull,(none),(none),130.4217,130.4217,0.0000,1.0000,Acceptable,NaN,NaN
1,PR_following_4W,Generalized gamma,(none),(none),130.4327,130.4327,0.0000,1.0000,Acceptable,NaN,NaN
2,PR_following_4W,Pearson type III,(none),(none),133.2201,133.2201,0.0000,1.0000,Acceptable,NaN,NaN
3,PR_following_4W,Gamma,(none),(none),134.7090,134.7090,0.0000,1.0000,Acceptable,NaN,NaN
4,PR_following_4W,Log-normal,Off_centeredness,Off_centeredness,138.6047,135.6874,2.9174,1.0000,Acceptable,NaN,NaN
5,PR_following_4W,Inverse Gaussian,Off_centeredness,Off_centeredness,138.8884,135.7687,3.1197,1.0000,Acceptable,NaN,NaN
6,PR_following_MT_3W,Weibull,Target_Speed_km/hr,Target_Speed_km/hr,278.6462,262.5678,16.0783,1.0000,Acceptable,NaN,NaN
7,PR_following_MT_3W,Generalized gamma,Target_Speed_km/hr,Target_Speed_km/hr,280.0712,264.0345,16.0367,1.0000,Acceptable,NaN,NaN
8,PR_following_MT_3W,Pearson type III,Target_Speed_km/hr,Target_Speed_km/hr,282.3901,265.2114,17.1787,1.0000,Acceptable,NaN,NaN
9,PR_following_MT_3W,Gamma,"Target_Speed_km/hr, Occupancy","Target_Speed_km/hr, Occupancy",287.7278,270.0170,17.7108,1.0440,Acceptable,NaN,NaN


,Pair,Distribution,Covariate,Covariate label,Coding,β,exp(β),Percent headway change,Effect direction,Drop-one LR χ²,Drop-one LR p,Final p < 0.05,VIF,VIF flag
0,PR_following_4W,Log-normal,Off_centeredness,Off-centeredness,"False=0, True=1; n0=11, n1=32",-0.3168,0.7285,-27.15,Shorter headway,4.917,0.0266,True,1.000,Acceptable
1,PR_following_4W,Inverse Gaussian,Off_centeredness,Off-centeredness,"False=0, True=1; n0=11, n1=32",-0.3285,0.7200,-28.00,Shorter headway,5.120,0.0237,True,1.000,Acceptable
2,PR_following_MT_3W,Weibull,Target_Speed_km/hr,Target Vehicle Speed,"Within-pair z-score; mean=11.4852, SD=3.59215",-0.1083,0.8974,-10.26,Shorter headway,18.078,0.0000,True,1.000,Acceptable
3,PR_following_MT_3W,Generalized gamma,Target_Speed_km/hr,Target Vehicle Speed,"Within-pair z-score; mean=11.4852, SD=3.59215",-0.1028,0.9023,-9.77,Shorter headway,18.037,0.0000,True,1.000,Acceptable
4,PR_following_MT_3W,Pearson type III,Target_Speed_km/hr,Target Vehicle Speed,"Within-pair z-score; mean=11.4852, SD=3.59215",-0.1085,0.8972,-10.28,Shorter headway,19.179,0.0000,True,1.000,Acceptable
5,PR_following_MT_3W,Gamma,Target_Speed_km/hr,Target Vehicle Speed,"Within-pair z-score; mean=11.4852, SD=3.59215",-0.1113,0.8947,-10.53,Shorter headway,12.342,0.0004,True,1.044,Acceptable
6,PR_following_MT_3W,Gamma,Occupancy,Occupancy,"False=0, True=1; n0=34, n1=70",-0.1627,0.8499,-15.01,Shorter headway,5.882,0.0153,True,1.044,Acceptable
7,PR_following_NMT_3W,Generalized gamma,Site,Site,"Shahjahanpur=0, Tikatuli=1; n0=14, n1=35",-0.2268,0.7971,-20.29,Shorter headway,4.311,0.0379,True,1.364,Acceptable
8,PR_following_NMT_3W,Generalized gamma,Occupancy,Occupancy,"False=0, True=1; n0=19, n1=30",0.1035,1.1091,10.91,Longer headway,4.274,0.0387,True,1.364,Acceptable
9,BTW_following_4W,Generalized gamma,Speed_Difference,Speed Difference,"Within-pair z-score; mean=-2.47466, SD=4.80557",-0.1152,0.8912,-10.88,Shorter headway,30.617,0.0000,True,1.012,Acceptable
